In [1]:
import sys
import os
sys.path.append(r'C:\Users\emira\OneDrive\Desktop\main projem')
os.chdir(r'C:\Users\emira\OneDrive\Desktop\main projem')

import numpy as np
import pandas as pd
import json
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

X_train = np.load('data/processed/X_train.npy')
X_test = np.load('data/processed/X_test.npy')
y_train = np.load('data/processed/y_train.npy')
y_test = np.load('data/processed/y_test.npy')

with open('data/processed/feature_names.json') as f:
    feature_names = json.load(f)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')


Train: (24987, 64), Test: (25013, 64)


In [2]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    verbosity=0
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"XGBoost: ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

XGBoost: ACC=0.9396, F1=0.9396, AUC=0.9867


In [3]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

xgb_base = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0)

search = RandomizedSearchCV(
    xgb_base, param_grid,
    n_iter=30,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
print(f"En iyi parametreler: {search.best_params_}")
print(f"CV Accuracy: {search.best_score_:.4f}")

Fitting 3 folds for each of 30 candidates, totalling 90 fits
En iyi parametreler: {'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.9}
CV Accuracy: 0.9400


In [4]:
best_xgb = XGBClassifier(
    subsample=0.8,
    n_estimators=300,
    min_child_weight=3,
    max_depth=8,
    learning_rate=0.05,
    gamma=0,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    verbosity=0
)

best_xgb.fit(X_train, y_train)
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Best XGBoost: ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

Best XGBoost: ACC=0.9408, F1=0.9409, AUC=0.9867


In [6]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    min_child_samples=20,
    num_leaves=63,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_test)
y_prob = lgbm.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"LightGBM: ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

C:\Users\emira\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emira\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM: ACC=0.9406, F1=0.9405, AUC=0.9869


In [7]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('xgb', best_xgb),
        ('lgbm', lgbm)
    ],
    voting='soft',
    n_jobs=-1
)

ensemble.fit(X_train, y_train)
y_pred = ensemble.predict(X_test)
y_prob = ensemble.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Ensemble (RF+XGB+LGBM): ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

C:\Users\emira\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Ensemble (RF+XGB+LGBM): ACC=0.9408, F1=0.9410, AUC=0.9874


C:\Users\emira\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [8]:
import math
import re

def add_new_features(X, feature_names, urls=None):
    new_features = []
    
    # Eğer URL listesi varsa URL bazlı feature ekle
    if urls is not None:
        for url in urls:
            url_str = str(url).lower()
            
            # 1. Entropy (rastgelelik skoru)
            freq = {}
            for c in url_str:
                freq[c] = freq.get(c, 0) + 1
            entropy = -sum((f/len(url_str)) * math.log2(f/len(url_str)) for f in freq.values() if f > 0)
            
            # 2. Consonant ratio
            consonants = sum(1 for c in url_str if c in 'bcdfghjklmnpqrstvwxyz')
            consonant_ratio = consonants / max(len(url_str), 1)
            
            # 3. Digit ratio
            digits = sum(1 for c in url_str if c.isdigit())
            digit_ratio = digits / max(len(url_str), 1)
            
            # 4. Special char ratio
            special = sum(1 for c in url_str if c in '-_@.=?&%+#~')
            special_ratio = special / max(len(url_str), 1)
            
            # 5. Subdomain count
            try:
                hostname = url_str.replace('https://','').replace('http://','').split('/')[0]
                subdomain_count = len(hostname.split('.')) - 2
            except:
                subdomain_count = 0
            
            new_features.append([entropy, consonant_ratio, digit_ratio, special_ratio, subdomain_count])
    
    return np.array(new_features)

# URL'leri yükle
raw_df = pd.read_csv('data/processed/raw_dataset.csv')
split_idx = int(len(raw_df) * 0.5)
train_urls = raw_df.iloc[:split_idx]['url'].values
test_urls = raw_df.iloc[split_idx:]['url'].values

print(f"Train URLs: {len(train_urls)}, Test URLs: {len(test_urls)}")

# Yeni feature'ları hesapla
new_train = add_new_features(None, None, train_urls)
new_test = add_new_features(None, None, test_urls)

print(f"Yeni feature shape: {new_train.shape}")
print(f"Örnek: entropy={new_train[0][0]:.3f}, consonant={new_train[0][1]:.3f}, digit={new_train[0][2]:.3f}")

Train URLs: 25000, Test URLs: 25000
Yeni feature shape: (25000, 5)
Örnek: entropy=3.995, consonant=0.552, digit=0.000


In [9]:
# Yeni feature'ları mevcut X'e ekle
X_train_new = np.hstack([X_train, new_train])
X_test_new = np.hstack([X_test, new_test])

print(f"Yeni X_train shape: {X_train_new.shape}")
print(f"Yeni X_test shape: {X_test_new.shape}")

# XGBoost ile dene
xgb_new = XGBClassifier(
    subsample=0.8,
    n_estimators=500,
    min_child_weight=3,
    max_depth=8,
    learning_rate=0.05,
    gamma=0,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    verbosity=0
)

xgb_new.fit(X_train_new, y_train)
y_pred = xgb_new.predict(X_test_new)
y_prob = xgb_new.predict_proba(X_test_new)[:,1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"XGBoost + 5 yeni feature (69 toplam): ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 24987 and the array at index 1 has size 25000

In [10]:
# Boyut eşitleme
min_train = min(X_train.shape[0], new_train.shape[0])
min_test = min(X_test.shape[0], new_test.shape[0])

X_train_new = np.hstack([X_train[:min_train], new_train[:min_train]])
X_test_new = np.hstack([X_test[:min_test], new_test[:min_test]])
y_train_new = y_train[:min_train]
y_test_new = y_test[:min_test]

print(f"X_train_new: {X_train_new.shape}")
print(f"X_test_new: {X_test_new.shape}")

xgb_new = XGBClassifier(
    subsample=0.8, n_estimators=500, min_child_weight=3,
    max_depth=8, learning_rate=0.05, gamma=0, colsample_bytree=0.9,
    random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0
)

xgb_new.fit(X_train_new, y_train_new)
y_pred = xgb_new.predict(X_test_new)
y_prob = xgb_new.predict_proba(X_test_new)[:,1]

acc = accuracy_score(y_test_new, y_pred)
f1 = f1_score(y_test_new, y_pred)
auc = roc_auc_score(y_test_new, y_prob)

print(f"XGBoost + 5 yeni feature (69 toplam): ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

X_train_new: (24987, 69)
X_test_new: (25000, 69)
XGBoost + 5 yeni feature (69 toplam): ACC=0.9388, F1=0.9389, AUC=0.9862


In [12]:
import joblib
import os

os.makedirs('models', exist_ok=True)
joblib.dump(best_xgb, 'models/best_xgboost.pkl')
print("Best XGBoost kaydedildi!")

print("\n=== FINAL MODEL SONUÇLARI ===")
print(f"Klasik RF (56 feature):         ACC=0.9336, F1=0.9340, AUC=0.9836")
print(f"Klasik + Cialdini (64 feature): ACC=0.9348, F1=0.9351, AUC=0.9838")
print(f"Best XGBoost (64 feature):      ACC=0.9408, F1=0.9409, AUC=0.9867")
print(f"Savunmalı model (saldırı altı): ACC=0.977")

Best XGBoost kaydedildi!

=== FINAL MODEL SONUÇLARI ===
Klasik RF (56 feature):         ACC=0.9336, F1=0.9340, AUC=0.9836
Klasik + Cialdini (64 feature): ACC=0.9348, F1=0.9351, AUC=0.9838
Best XGBoost (64 feature):      ACC=0.9408, F1=0.9409, AUC=0.9867
Savunmalı model (saldırı altı): ACC=0.977


In [13]:
print(f"Train phishing: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Train meşru: {(1-y_train).sum()} ({(1-y_train).mean()*100:.1f}%)")

Train phishing: 12435 (49.8%)
Train meşru: 12552 (50.2%)


In [14]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# Önemli feature'ları seç
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_train, y_train)

# En önemli 30 feature
selector = SelectFromModel(rf_selector, max_features=30, prefit=True)
X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

selected_features = [feature_names[i] for i in range(len(feature_names)) if selector.get_support()[i]]
print(f"Seçilen feature sayısı: {X_train_sel.shape[1]}")
print(f"Seçilen feature'lar: {selected_features}")

# XGBoost ile dene
xgb_sel = XGBClassifier(
    subsample=0.8, n_estimators=500, min_child_weight=3,
    max_depth=8, learning_rate=0.05, gamma=0, colsample_bytree=0.9,
    random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0
)
xgb_sel.fit(X_train_sel, y_train)
y_pred = xgb_sel.predict(X_test_sel)
y_prob = xgb_sel.predict_proba(X_test_sel)[:,1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
print(f"\nXGBoost + Feature Selection: ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

Seçilen feature sayısı: 22
Seçilen feature'lar: ['url_length', 'domain_length', 'path_length', 'subdomain_length', 'num_subdomains', 'num_dots', 'num_hyphens', 'num_slashes', 'digit_ratio', 'alpha_ratio', 'special_ratio', 'url_entropy', 'domain_entropy', 'path_entropy', 'is_https', 'tld_trusted', 'tld_length', 'suspicious_keywords', 'path_depth', 'min_brand_levenshtein', 'min_brand_levenshtein_norm', 'cld_commitment']

XGBoost + Feature Selection: ACC=0.9344, F1=0.9344, AUC=0.9847


In [15]:
import joblib

# XGBoost'u src/utils'in okuyabileceği formatta kaydet
joblib.dump(best_xgb, 'models/best_xgboost.pkl')
print("Kaydedildi!")
print(f"Dosya var mı: {os.path.exists('models/best_xgboost.pkl')}")

Kaydedildi!
Dosya var mı: True
